# 102 · Framework: Patterns of Scientific Logic

In the previous tutorial, we manually interacted with the `Ledger`. While powerful, manual event management becomes complex as your scientific logic grows. 

The `EarlySign` **Framework layer** introduces the **Pattern G** architecture, which provides core abstractions to manage the lifecycle and lineage of an analysis:

1. **Trace & Lineage**: The statistical "provenance" that links results to raw data.
2. **Projectors**: Pure functional views that translate history into scientific objects.
3. **Entities**: Identifiable aggregates that support efficient snapshotting.
4. **Sessions & Writers**: Orchestrators that handle reading, writing, and lineage tracking.

In this tutorial, we will explore these components by building a simple "Counter" system.

## 1. Setup

We initialize a Ledger as before.

In [ ]:
import json
from typing import Optional

import ibis
from pydantic import BaseModel

from earlysign.core.ledger import Ledger
from earlysign.v1.framework.entity.core import Entity
from earlysign.v1.framework.entity.snapshot import Snapshot
from earlysign.v1.framework.projector import ProjectionResult, Projector
from earlysign.v1.framework.session import Session
from earlysign.v1.framework.trace import Traced, TraceId

con = ibis.connect("duckdb://:memory:")
ledger = Ledger(con, "framework_demo").bind(experiment_id="102_demo")
ledger.ensure()

## 2. Scientific Provenance: The Trace

In science, a result is only as good as its provenance. `EarlySign` uses a **Trace**—a list of parent record UUIDs—to track exactly which inputs influenced a computation.

- **`TraceId`**: A type-safe identifier for a record UUID.
- **`Traced[T]`**: A generic container that pairs any data `T` with its `trace`.

This forms an **Implicit Web of Proof**: every committed fact in the ledger points back to the events that generated it.

In [ ]:
# Example: A manually traced value
val = Traced(data=42, trace=[TraceId("abc-123")])
print(f"Data: {val.data}, Provenance: {val.trace}")

## 3. Pure Functional Views: Projectors

A **`Projector`** is a lens into the ledger. It takes a stream of events and "projects" them into a meaningful object. Because it is functional, it can be replayed at any time. 

Projectors always return a **`ProjectionResult`** (a subclass of `Traced`), ensuring lineage is never lost.

In [ ]:
class CounterState(BaseModel):
    total: int = 0


class IncrementProjector(Projector[CounterState]):
    """Reads ALL increments to get current state."""

    def project(self, table: ibis.Expr) -> ProjectionResult[CounterState]:
        # 1. Select relevant events
        matches = table.filter(table.type == "Increment")
        pdf = matches.execute()

        if pdf.empty:
            return ProjectionResult(data=CounterState(total=0), trace=[])

        # 2. Derive state
        total = (
            pdf["payload"]
            .apply(
                lambda x: json.loads(x)["value"] if isinstance(x, str) else x["value"]
            )
            .sum()
        )

        # 3. Collect lineage (trace)
        trace = [TraceId(str(u)) for u in pdf["uuid"]]

        return ProjectionResult(data=CounterState(total=total), trace=trace)

## 4. Identifiable Aggregates: Entities

While Projectors process the entire history, **`Entity`** provides a way to manage objects with **consistent identity** and **snapshots**. This is essential for efficiency when dealing with millions of events.

An Entity automatically:
1. Finds its latest **Snapshot**.
2. Identifies only the **Delta** (events since the snapshot).
3. Folds the delta into the snapshot using its `compute()` logic.

In [ ]:
class CounterEntity(Entity[CounterState]):
    """An identifiable counter that uses snapshots for efficiency."""

    data_type = CounterState

    @property
    def initial_value(self) -> CounterState:
        return CounterState(total=0)

    def compute(
        self,
        snapshot: Optional[Snapshot[CounterState]],
        delta_expr: ibis.Expr,
        full_table: ibis.Expr,
    ) -> ProjectionResult[CounterState]:
        current_total = snapshot.data.total if snapshot else 0

        # Process only the delta (new increments)
        matches = delta_expr.filter(delta_expr.type == "Increment")
        pdf = matches.execute()

        for _, row in pdf.iterrows():
            p = row["payload"]
            val = json.loads(p)["value"] if isinstance(p, str) else p["value"]
            current_total += val

        # The trace combines the snapshot and any new events
        trace = [TraceId(str(u)) for u in pdf["uuid"]]
        if snapshot and snapshot.uuid:
            trace.insert(0, TraceId(str(snapshot.uuid)))

        return ProjectionResult(data=CounterState(total=current_total), trace=trace)

## 5. Orchestration: Sessions & Writers

The **`Session`** brings everything together. It acts as an orchestrator that tracks lineage across multiple reads and writes.

- `sess.read()`: Evokes a Projector or Entity and merges its trace into the session.
- `sess.commit()`: Writes a fact to the ledger, automatically tagging it with the accumulated session trace.
- `entity.save()`: Commits a new snapshot for an entity.

In [ ]:
class Increment(BaseModel):
    value: int


class Summary(BaseModel):
    text: str


# 1. Initial State
ledger.insert(Increment(value=10))
ledger.insert(Increment(value=5))

with Session(ledger) as sess:
    # 2. Read using the efficient Entity pattern
    counter = CounterEntity(identity="my_first_counter")
    result = sess.read(counter)

    print(f"Current Total: {result.data.total}")

    # 3. Save a snapshot to speed up future reads
    counter.save(sess, result)

    # 4. Commit a high-level summary
    sess.commit(Summary(text="Initial count completed"))

print("Session done. Check the ledger to see the snapshots and traces.")
display(ledger.t.execute())

## 6. Summary

- **Trace**: The fundamental unit of scientific causality.
- **Projectors**: Pure logic for replaying history.
- **Entities**: Identifiable aggregates that use snapshots for performance.
- **Sessions**: Orchestrators that weave the "web of proof".

In the next section, we will see how these components power real-world **Group Sequential Designs**.